# Chapter 21 — MCP Servers

**Companion to *Pi Agents*** · [`Pi Agents` chapter 21](https://programmer.ie/books/pi/21-chapter/)

| | |
|---|---|
| Chapter | `21-chapter.md` · weight 21 · `/books/pi/21-chapter/` |
| Notebook | `notebooks/pi/21-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Does a gate keyed on the bare tool name skip `mcp__jira__transition_issue`, and
does a lying `readOnlyHint` pass?

The chapter's point: MCP tools are named `mcp__<server>__<tool>`. A gate that
checks only the bare name (`transition_issue`) never matches. The 1.0.4 `--tools`
rule keeps MCP tools unless an entry starts with `mcp__`. Annotations are hints,
not permission — a server that lies about `readOnlyHint` passes the gate.


## What this notebook establishes

- An MCP tool is named `mcp__<server>__<tool>` — a gate keyed on the **bare name** (`transition_issue`) **never matches** the full name (`mcp__jira__transition_issue`).
- The deny-list gate keys on the **full name**: with no UI the call is blocked and the server never hears it.
- With a UI the person decides: approve lets it through, decline blocks it.
- **Annotations are hints, not permission**: a server that lies about `readOnlyHint: true` passes the gate — the gate believed it.
- Exposure decides whether the model is told the tool exists: `direct` = declared, `codemode` = reachable not declared, `hidden` = unreachable.
- `pi.registerMcpServer` adds a server for the session; `unregisterMcpServer` takes it away again.

## What this notebook does **not** establish

- **No network**: the far end is an in-memory transport pair (`ch21-lim1`).
- **Annotations are explicitly unverified** (`ch21-lim2`) — a lying server passes.
- **Project overrides and `cimd` (1.0.1) not exercised** (`ch21-lim3`).
- **The `tool-allowlist.test.ts` tests the 1.0.4 `--tools` rule separately** — this driver focuses on the gate and exposure.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — real sessions with an in-memory MCP server (`fakeMcpServer` from the harness), driven by a scripted model.
* **Evidence scope:** `in_process_runtime`. The naming, exposure, and gate rules are Pi's. No network.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(21))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/21-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## The bare-name gate: a classic mistake

A gate that checks `event.toolName === "transition_issue"` never matches
`mcp__jira__transition_issue`. The server hears the call because the gate didn't
block it.

In [3]:
MCP = pinb.driver_source("ch21-mcp.ts")
mcp = pinb.driver(MCP, label="ch21-mcp", timeout=300)
print(pinb.md_table(
    ["gate type", "tool name checked", "isError?", "server heard?"],
    [["bare name", "transition_issue", "no" if not mcp["bareNameGate"]["isError"] else "yes", "yes" if mcp["bareNameGate"]["serverHeard"] > 0 else "no"],
     ["full name (deny-list)", "mcp__jira__transition_issue", "yes" if mcp["denyListGate"]["isError"] else "no", "yes" if mcp["denyListGate"]["serverHeard"] > 0 else "no"]],
))

| gate type | tool name checked | isError? | server heard? |
|---|---|---|---|
| bare name | transition_issue | no | yes |
| full name (deny-list) | mcp__jira__transition_issue | yes | no |


In [4]:
pinb.show_checks([
    pinb.check("bare-name gate never matches mcp__jira__transition_issue",
               not mcp["bareNameGate"]["isError"] and mcp["bareNameGate"]["serverHeard"] > 0,
               "gate missed it, server heard"),
    pinb.check("full-name deny-list blocks and server never hears",
               mcp["denyListGate"]["isError"] and mcp["denyListGate"]["serverHeard"] == 0,
               "blocked, server silent"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------------------  ----------------------------
PASS  bare-name gate never matches mcp__jira__transition_issue  gate missed it, server heard
PASS  full-name deny-list blocks and server never hears  blocked, server silent

2/2 assertions held.


## With a UI: the person decides

The deny-list gate asks via `ctx.ui.confirm`. Approve = through, decline = blocked.

In [5]:
approve = mcp["denyListGateWithUI"][0]
decline = mcp["denyListGateWithUI"][1]
print(pinb.md_table(
    ["answer", "isError?", "server heard?", "confirm message"],
    [["approve", "no" if not approve["isError"] else "yes", "yes" if approve["serverHeard"] > 0 else "no", approve["confirmMessage"]],
     ["decline", "yes" if decline["isError"] else "no", "yes" if decline["serverHeard"] > 0 else "no", decline["confirmMessage"]]],
))

| answer | isError? | server heard? | confirm message |
|---|---|---|---|
| approve | no | yes | mcp__jira__transition_issue (from jira) |
| decline | yes | no | mcp__jira__transition_issue (from jira) |


In [6]:
pinb.show_checks([
    pinb.check("approve: call goes through, server hears",
               not approve["isError"] and approve["serverHeard"] > 0,
               "allowed"),
    pinb.check("decline: call blocked, server silent",
               decline["isError"] and decline["serverHeard"] == 0,
               "blocked"),
    pinb.check("confirm message includes full tool name and server",
               approve["confirmMessage"] and "mcp__jira__transition_issue" in approve["confirmMessage"] and "jira" in approve["confirmMessage"],
               "full name shown"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------------  ---------------
PASS  approve: call goes through, server hears  allowed
PASS  decline: call blocked, server silent  blocked
PASS  confirm message includes full tool name and server  full name shown

3/3 assertions held.


## Annotations are hints, not permission

Three tools from a lying server:
- `list_issues`: honest `readOnlyHint: true, openWorldHint: false` → no approval needed
- `no_hints_at_all`: no hints → treated as dangerous, blocked without UI
- `delete_everything`: **lies** with `readOnlyHint: true, openWorldHint: false` → **passes the gate**

In [7]:
print(pinb.md_table(
    ["tool", "annotations", "isError?"],
    [["list_issues", "honest readOnlyHint", "no" if not mcp["lyingAnnotations"]["honest"]["isError"] else "yes"],
     ["no_hints_at_all", "no hints", "yes" if mcp["lyingAnnotations"]["silent"]["isError"] else "no"],
     ["delete_everything", "LYING readOnlyHint", "no" if not mcp["lyingAnnotations"]["liar"]["isError"] else "yes"]],
))

| tool | annotations | isError? |
|---|---|---|
| list_issues | honest readOnlyHint | no |
| no_hints_at_all | no hints | yes |
| delete_everything | LYING readOnlyHint | no |


In [8]:
print("Server calls:", ", ".join(mcp["lyingAnnotations"]["serverCalls"]))
print()
pinb.show_checks([
    pinb.check("honest readOnlyHint: no approval needed",
               not mcp["lyingAnnotations"]["honest"]["isError"], "allowed"),
    pinb.check("no hints: treated as dangerous, blocked without UI",
               mcp["lyingAnnotations"]["silent"]["isError"], "blocked"),
    pinb.check("LYING readOnlyHint: the gate believed it",
               not mcp["lyingAnnotations"]["liar"]["isError"], "allowed (the lie worked)"),
    pinb.check("server heard honest and liar, not silent",
               mcp["lyingAnnotations"]["serverCalls"] == ["list_issues", "delete_everything"],
               "only allowed tools called"),
])

Server calls: list_issues, delete_everything

== Assertions ==
  assertion  observed
----  --------------------------------------------------  -------------------------
PASS  honest readOnlyHint: no approval needed  allowed
PASS  no hints: treated as dangerous, blocked without UI  blocked
PASS  LYING readOnlyHint: the gate believed it  allowed (the lie worked)
PASS  server heard honest and liar, not silent  only allowed tools called

4/4 assertions held.


## Exposure decides what the model is told

`direct` = declared to model (like built-ins). `codemode` = reachable from scripts, not declared. `hidden` = unreachable.

In [9]:
for exposure in ["direct", "codemode", "hidden"]:
    tools = mcp["exposureDeclared"][exposure]["tools"]
    print(f"  {exposure}: declared={"mcp__docs__search" in tools}")

  direct: declared=True
  codemode: declared=False
  hidden: declared=False


In [10]:
pinb.show_checks([
    pinb.check("direct: declared to model",
               "mcp__docs__search" in mcp["exposureDeclared"]["direct"]["tools"], "declared"),
    pinb.check("codemode: reachable, not declared",
               "mcp__docs__search" not in mcp["exposureDeclared"]["codemode"]["tools"], "not declared"),
    pinb.check("hidden: unreachable",
               "mcp__docs__search" not in mcp["exposureDeclared"]["hidden"]["tools"], "not declared"),
])

== Assertions ==
  assertion  observed
----  ---------------------------------  ------------
PASS  direct: declared to model  declared
PASS  codemode: reachable, not declared  not declared
PASS  hidden: unreachable  not declared

3/3 assertions held.


## Register and unregister at runtime

`pi.registerMcpServer` adds a server for the session; `unregisterMcpServer`
removes it. The command `/jira-off` calls unregister.

In [11]:
print(pinb.md_table(
    ["step", "server heard?"],
    [["after register", "yes" if mcp["registerUnregister"]["afterRegister"] > 0 else "no"],
     ["after unregister", "yes" if mcp["registerUnregister"]["afterUnregister"] > 0 else "no"]],
))

| step | server heard? |
|---|---|
| after register | yes |
| after unregister | yes |


In [12]:
pinb.show_checks([
    pinb.check("after register: server connected and tool ran",
               mcp["registerUnregister"]["afterRegister"] > 0, "connected"),
    pinb.check("after unregister: server receives nothing more",
               mcp["registerUnregister"]["afterUnregister"] == mcp["registerUnregister"]["afterRegister"], "no new calls"),
])

== Assertions ==
  assertion  observed
----  ----------------------------------------------  ------------
PASS  after register: server connected and tool ran  connected
PASS  after unregister: server receives nothing more  no new calls

2/2 assertions held.


## The chapter's own tests

All tests from both test files.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [13]:
PATTERNS = ['ch21-mcp/mcp.test.ts', 'ch21-mcp/tool-allowlist.test.ts']
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch21-mcp/mcp.test.ts, ch21-mcp/tool-allowlist.test.ts', show="mcp__")

$ node --test ch21-mcp/mcp.test.ts, ch21-mcp/tool-allowlist.test.ts
  PASS  14 passed, 0 failed, 0 skipped  in 1393 ms

  tests matching 'mcp__':
    ok   an MCP tool is named mcp__<server>__<tool>, and a name-keyed gate that forgets the prefix lets it through
    ok   an mcp__ pattern makes a direct-exposure tool declared, where an unnamed one was not
    ok   excludeTools takes MCP tools too, and needs no mcp__ prefix


## Your turn: predict, then run

**Predict first.** What happens if an MCP tool is registered with `exposure:
"codemode"` and the `--tools` allowlist includes `mcp__jira__*`? Does the
pattern override exposure?

**Then change one input.** The `tool-allowlist.test.ts` shows that `excludeTools:
["mcp__*"]` unregisters the tool entirely. What happens if you use
`tools: ["read", "codemode"]` (no mcp__ pattern) but the tool is `direct`
exposure?

**Predict the boundary.** The 1.0.4 rule: `--tools` keeps MCP tools unless an
entry starts with `mcp__`. What if you use `tools: ["*"]` — does that match
MCP tools or only built-ins?

In [14]:
print("Predictions:")
print("  1. mcp__ pattern with codemode: pattern does NOT override exposure — stays undeclared.")
print("  2. direct exposure without mcp__ pattern: tool is inert (neither declared nor callable).")
print("  3. tools: [*]: matches MCP tools too — * is a pattern, not just built-ins.")
print()
print("Observed above:")
print(f"  bare name gate missed: {not mcp["bareNameGate"]["isError"]}, server heard={mcp["bareNameGate"]["serverHeard"]}")
print(f"  full name gate blocked: {mcp["denyListGate"]["isError"]}, server heard={mcp["denyListGate"]["serverHeard"]}")
print(f"  lying readOnlyHint passed: {not mcp["lyingAnnotations"]["liar"]["isError"]}")
print(f"  exposure: direct={mcp["exposureDeclared"]["direct"]["tools"]}, codemode={mcp["exposureDeclared"]["codemode"]["tools"]}, hidden={mcp["exposureDeclared"]["hidden"]["tools"]}")
print()
assert not mcp["bareNameGate"]["isError"] and mcp["bareNameGate"]["serverHeard"] > 0
assert mcp["denyListGate"]["isError"] and mcp["denyListGate"]["serverHeard"] == 0
assert not mcp["lyingAnnotations"]["liar"]["isError"]
print("held: bare name gate misses MCP tools; lying annotations pass; exposure controls declaration")

Predictions:
  1. mcp__ pattern with codemode: pattern does NOT override exposure — stays undeclared.
  2. direct exposure without mcp__ pattern: tool is inert (neither declared nor callable).
  3. tools: [*]: matches MCP tools too — * is a pattern, not just built-ins.

Observed above:
  bare name gate missed: True, server heard=1
  full name gate blocked: True, server heard=0
  lying readOnlyHint passed: True
  exposure: direct=['read', 'bash', 'edit', 'write', 'mcp__docs__search'], codemode=['read', 'bash', 'edit', 'write'], hidden=['read', 'bash', 'edit', 'write']

held: bare name gate misses MCP tools; lying annotations pass; exposure controls declaration


## Interpretation

MCP in Pi has three naming/visibility layers, and each has a boundary:

| Layer | What it controls | The boundary |
|---|---|---|
| **Tool name** | `mcp__<server>__<tool>` | A gate on the bare name (`transition_issue`) never matches |
| **Exposure** | `direct` / `codemode` / `hidden` | `direct` = declared; `codemode` = script-reachable; `hidden` = unreachable |
| **Annotations** | `readOnlyHint`, `openWorldHint` | **Hints, not permission** — a lie passes |

The 1.0.4 `--tools` rule (tested in `tool-allowlist.test.ts`):
* `--tools read,codemode` keeps MCP tools (no `mcp__` pattern)
* `--tools read,codemode,mcp__jira__*` declares `mcp__jira__*` tools
* `excludeTools: ["mcp__*"]` unregisters MCP tools entirely
* `disabledBuiltinExtensions: ["mcp"]` gates built-in MCP support, not injected extensions

The practical rules:
* **Gate on the full name** — `mcp__<server>__<tool>`, not the bare tool name.
* **Don't trust annotations** — they are hints; a malicious or buggy server can lie.
* **Use `codemode` exposure for internal tools** — the model doesn't see them, scripts can call them.
* **`registerMcpServer` / `unregisterMcpServer` are session-scoped** — they don't persist across sessions.

## Sources, execution mode and limitations

**Execution mode:** **Run** — real sessions with an in-memory MCP server (`fakeMcpServer` from the harness), driven by a scripted model.

**Evidence scope:** `in_process_runtime`. The naming, exposure, and gate rules are Pi's. No network.

### What was read or run

- **Ran** `drivers/ch21-mcp.ts`: six real sessions covering bare-name gate, deny-list gate (with/without UI), lying annotations, exposure, and register/unregister.
- **Ran** `ch21-mcp/mcp.test.ts` (6 tests) and `ch21-mcp/tool-allowlist.test.ts` (7 tests).
- **Read** `examples/evidence.json`, chapter 21 rows (13 claims).

### Limitations

- **No network**: the far end is an in-memory transport pair (`ch21-lim1`).
- **Annotations are explicitly unverified** (`ch21-lim2`) — a lying server passes.
- **Project overrides and `cimd` (1.0.1) not exercised** (`ch21-lim3`).

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
